# 💰 TP4 — Prédire la Facture Mensuelle d'un Client Télécom (Régression)
## Module Data Science — Machine Learning | Corrigé détaillé
**Dataset** : `clients_telecom.csv` (8 000 clients) — cible : `facture_mensuelle` (FCFA)

In [ ]:
!pip install scikit-learn seaborn -q
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score
sns.set_theme(style='whitegrid')

## 1. Exploration (EDA)

In [ ]:
df = pd.read_csv("clients_telecom.csv")
print("Dimensions :", df.shape)
df.head()

In [ ]:
df.info()
print(df["facture_mensuelle"].describe())
plt.figure(figsize=(8,5)); sns.histplot(df["facture_mensuelle"], bins=50, kde=True)
plt.title("Distribution de la facture"); plt.xlabel("Facture (FCFA)"); plt.show()

In [ ]:
plt.figure(figsize=(8,6))
cols = ["age","anciennete_mois","conso_data_go","minutes_appel","nb_sms","facture_mensuelle"]
sns.heatmap(df[cols].corr(), annot=True, cmap="coolwarm", center=0, fmt=".2f")
plt.title("Corrélations"); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14,5))
sns.scatterplot(data=df, x="conso_data_go", y="facture_mensuelle", alpha=0.3, ax=axes[0])
axes[0].set_title("Data vs Facture")
sns.scatterplot(data=df, x="minutes_appel", y="facture_mensuelle", alpha=0.3, ax=axes[1])
axes[1].set_title("Minutes vs Facture")
plt.tight_layout(); plt.show()

> 🔑 `conso_data_go` et `minutes_appel` très corrélées à la facture (postes principaux).

## 2. Préparation (exclure client_id et la cible)

In [ ]:
features = ["age", "anciennete_mois", "conso_data_go", "minutes_appel",
            "nb_sms", "nb_reclamations"]
X = df[features]
y = df["facture_mensuelle"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)
print(f"Train : {X_train.shape[0]} | Test : {X_test.shape[0]}")

> ⚠️ Le KNN exige la normalisation. Régression linéaire et arbre : données brutes.

## 3. Les 3 modèles de régression

In [ ]:
lr    = LinearRegression().fit(X_train, y_train)
arbre = DecisionTreeRegressor(max_depth=8, random_state=42).fit(X_train, y_train)
knn   = KNeighborsRegressor(n_neighbors=5).fit(X_train_s, y_train)
pred_lr, pred_arbre, pred_knn = lr.predict(X_test), arbre.predict(X_test), knn.predict(X_test_s)
print("✅ 3 modèles entraînés")

## 4. Évaluation (R², MAE, overfitting)

In [ ]:
for nom, pred in [("Régression linéaire", pred_lr),
                  ("Arbre de décision", pred_arbre),
                  ("KNN (k=5)", pred_knn)]:
    print(f"{nom:22} | R² = {r2_score(y_test, pred):.3f} | "
          f"MAE = {mean_absolute_error(y_test, pred):,.0f} FCFA")

In [ ]:
plt.figure(figsize=(8,8))
plt.scatter(y_test, pred_lr, alpha=0.3)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--", lw=2)
plt.xlabel("Facture réelle"); plt.ylabel("Facture prédite")
plt.title("Prédictions vs Réalité"); plt.show()

In [ ]:
print("Effet de la profondeur de l'arbre :")
for prof in [3, 5, 8, 12, None]:
    a = DecisionTreeRegressor(max_depth=prof, random_state=42).fit(X_train, y_train)
    print(f"  max_depth={str(prof):5} → R² = {r2_score(y_test, a.predict(X_test)):.3f}")

## 5. Réflexion (Q13) — Pourquoi le R² est-il si élevé (~0.99) ?

La facture est **calculée** à partir des consommations (`data × prix + minutes × prix + SMS × prix + abonnement`) → relation quasi **déterministe** que le modèle retrouve presque parfaitement.

> 🔑 Un R² proche de 1 signale soit un problème « facile » (ici), soit une **fuite de données** — toujours se demander pourquoi.

## 6. Interpréter et prédire

In [ ]:
importances = pd.Series(arbre.feature_importances_, index=features).sort_values(ascending=False)
plt.figure(figsize=(9,5)); importances.plot(kind="barh")
plt.title("Variables importantes"); plt.gca().invert_yaxis(); plt.show()
print(importances.head(5))

In [ ]:
def predire_facture(modele, colonnes, **infos):
    client = pd.DataFrame(0, index=[0], columns=colonnes)
    for cle, val in infos.items():
        if cle in client.columns:
            client[cle] = val
    return modele.predict(client)[0]

f1 = predire_facture(lr, features, age=28, anciennete_mois=24,
                     conso_data_go=40, minutes_appel=300, nb_sms=100, nb_reclamations=0)
print(f"Gros consommateur data → {f1:,.0f} FCFA")
f2 = predire_facture(lr, features, age=55, anciennete_mois=60,
                     conso_data_go=3, minutes_appel=80, nb_sms=15, nb_reclamations=1)
print(f"Petit consommateur → {f2:,.0f} FCFA")

## 7. Question bonus — Temps de prédiction

In [ ]:
import time
for nom, modele, X_eval in [("Régression linéaire", lr, X_test),
                            ("Arbre de décision", arbre, X_test),
                            ("KNN (k=5)", knn, X_test_s)]:
    t0 = time.time(); modele.predict(X_eval)
    print(f"{nom:22} → {time.time()-t0:.4f} s")
# → le KNN est le plus lent (algorithme paresseux : calcule les distances à la prédiction)

## 8. Conclusion

Meilleur modèle : **régression linéaire** (R² ~0.99, MAE ~1 570 FCFA). R² très élevé car la facture est calculée à partir des consommations. Variables clés : `conso_data_go`, `minutes_appel`.